# Credit Card Fraud Detection Using Deep Learning
Run cells from top to bottom in Google Colab. Upload `creditcard.csv` when prompted.
This notebook implements MLP, 1D CNN and Autoencoder; BCE, class-weighted BCE and Focal Loss experiments; validation threshold selection; test evaluation; and optional SHAP/LIME explanations.

## 1. Install packages

In [ ]:
!pip -q install shap lime

## 2. Imports and reproducibility

In [ ]:
import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_curve)

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 3. Upload and load dataset

In [ ]:
from google.colab import files
import pandas as pd

# Upload the actual creditcard.csv file (not archive.zip)
uploaded = files.upload()

csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise ValueError('Please upload creditcard.csv as a .csv file, not a ZIP archive.')

# Prefer the expected dataset filename if more than one CSV is uploaded.
csv_name = next((name for name in csv_files if 'creditcard' in name.lower()), csv_files[0])
df = pd.read_csv(csv_name)

required_columns = {'Class', 'Time', 'Amount'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'This does not look like the expected credit card dataset. Missing columns: {missing_columns}')

print('Dataset loaded successfully!')
print('File:', csv_name)
print('Dataset shape:', df.shape)
display(df.head())


## 4. Inspect dataset and class imbalance

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["Class"].value_counts())
print((df["Class"].value_counts(normalize=True)*100).round(4))
sns.countplot(data=df, x="Class")
plt.title("Class distribution: 0 = genuine, 1 = fraud")
plt.show()
assert "Class" in df.columns and df.isnull().sum().sum()==0

## 5. Split into training, validation and test sets

In [ ]:
X=df.drop(columns=["Class"]).copy()
y=df["Class"].astype("int32").copy()
X_train, X_temp, y_train, y_temp=train_test_split(
    X,y,test_size=0.40,stratify=y,random_state=SEED)
X_val, X_test, y_val, y_test=train_test_split(
    X_temp,y_temp,test_size=0.50,stratify=y_temp,random_state=SEED)
print("Train",X_train.shape,"fraud",int(y_train.sum()))
print("Validation",X_val.shape,"fraud",int(y_val.sum()))
print("Test",X_test.shape,"fraud",int(y_test.sum()))

## 6. Scale Time and Amount using training data only

In [ ]:
feature_names=X_train.columns.tolist()
scaler=StandardScaler()
scale_cols=["Time","Amount"]
X_train.loc[:,scale_cols]=scaler.fit_transform(X_train[scale_cols])
X_val.loc[:,scale_cols]=scaler.transform(X_val[scale_cols])
X_test.loc[:,scale_cols]=scaler.transform(X_test[scale_cols])
X_train_np=X_train.to_numpy(dtype=np.float32)
X_val_np=X_val.to_numpy(dtype=np.float32)
X_test_np=X_test.to_numpy(dtype=np.float32)
y_train_np=y_train.to_numpy(dtype=np.float32)
y_val_np=y_val.to_numpy(dtype=np.int32)
y_test_np=y_test.to_numpy(dtype=np.int32)
print("Input feature count:",X_train_np.shape[1])

## 7. Calculate class weights

In [ ]:
w=compute_class_weight(class_weight="balanced",classes=np.array([0,1]),y=y_train_np.astype(int))
class_weights={0:float(w[0]),1:float(w[1])}
print(class_weights)

## 8. Define loss functions and deep learning architectures

In [ ]:
def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true,y_pred):
        yt=tf.cast(tf.reshape(y_true,(-1,1)),tf.float32)
        yp=tf.clip_by_value(tf.cast(y_pred,tf.float32),1e-7,1-1e-7)
        pt=yt*yp+(1-yt)*(1-yp)
        at=yt*alpha+(1-yt)*(1-alpha)
        return tf.reduce_mean(-at*tf.pow(1-pt,gamma)*tf.math.log(pt))
    return loss

def build_mlp(n):
    return tf.keras.Sequential([
        tf.keras.layers.Input((n,)),
        tf.keras.layers.Dense(64,activation="relu"),
        tf.keras.layers.Dropout(.30),
        tf.keras.layers.Dense(32,activation="relu"),
        tf.keras.layers.Dropout(.20),
        tf.keras.layers.Dense(1,activation="sigmoid")])

def build_cnn(shape):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape),
        tf.keras.layers.Conv1D(64,3,padding="same",activation="relu"),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(32,3,padding="same",activation="relu"),
        tf.keras.layers.GlobalMaxPooling1D(),
        tf.keras.layers.Dense(32,activation="relu"),
        tf.keras.layers.Dropout(.30),
        tf.keras.layers.Dense(1,activation="sigmoid")])

def compile_model(model,loss="binary_crossentropy"):
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss=loss)
    return model

def get_callbacks():
    return [tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=3,restore_best_weights=True)]

EPOCHS=15
BATCH_SIZE=512

## 9. Train MLP with standard Binary Cross-Entropy (baseline)

In [ ]:
mlp_bce=compile_model(build_mlp(X_train_np.shape[1]))
hist_mlp_bce=mlp_bce.fit(X_train_np,y_train_np,validation_data=(X_val_np,y_val_np),
    epochs=EPOCHS,batch_size=BATCH_SIZE,callbacks=get_callbacks(),verbose=1)

## 10. Train MLP with class-weighted BCE

In [ ]:
mlp_weighted=compile_model(build_mlp(X_train_np.shape[1]))
hist_mlp_weighted=mlp_weighted.fit(X_train_np,y_train_np,validation_data=(X_val_np,y_val_np),
    epochs=EPOCHS,batch_size=BATCH_SIZE,class_weight=class_weights,
    callbacks=get_callbacks(),verbose=1)

## 11. Train MLP with Focal Loss

In [ ]:
mlp_focal=compile_model(build_mlp(X_train_np.shape[1]),focal_loss())
hist_mlp_focal=mlp_focal.fit(X_train_np,y_train_np,validation_data=(X_val_np,y_val_np),
    epochs=EPOCHS,batch_size=BATCH_SIZE,callbacks=get_callbacks(),verbose=1)

## 12. Prepare CNN input and train CNN with BCE

In [ ]:
X_train_cnn=X_train_np.reshape(-1,X_train_np.shape[1],1)
X_val_cnn=X_val_np.reshape(-1,X_val_np.shape[1],1)
X_test_cnn=X_test_np.reshape(-1,X_test_np.shape[1],1)
cnn_bce=compile_model(build_cnn((X_train_cnn.shape[1],1)))
hist_cnn_bce=cnn_bce.fit(X_train_cnn,y_train_np,validation_data=(X_val_cnn,y_val_np),
    epochs=EPOCHS,batch_size=BATCH_SIZE,callbacks=get_callbacks(),verbose=1)

## 13. Train CNN with class-weighted BCE

In [ ]:
cnn_weighted=compile_model(build_cnn((X_train_cnn.shape[1],1)))
hist_cnn_weighted=cnn_weighted.fit(X_train_cnn,y_train_np,validation_data=(X_val_cnn,y_val_np),
    epochs=EPOCHS,batch_size=BATCH_SIZE,class_weight=class_weights,
    callbacks=get_callbacks(),verbose=1)

## 14. Train CNN with Focal Loss

In [ ]:
cnn_focal=compile_model(build_cnn((X_train_cnn.shape[1],1)),focal_loss())
hist_cnn_focal=cnn_focal.fit(X_train_cnn,y_train_np,validation_data=(X_val_cnn,y_val_np),
    epochs=EPOCHS,batch_size=BATCH_SIZE,callbacks=get_callbacks(),verbose=1)

## 15. Train Autoencoder on genuine transactions

In [ ]:
inp=tf.keras.layers.Input(shape=(X_train_np.shape[1],))
z=tf.keras.layers.Dense(16,activation="relu")(inp)
z=tf.keras.layers.Dense(8,activation="relu")(z)
z=tf.keras.layers.Dense(16,activation="relu")(z)
out=tf.keras.layers.Dense(X_train_np.shape[1],activation="linear")(z)
autoencoder=tf.keras.Model(inp,out)
autoencoder.compile(optimizer="adam",loss="mse")
X_train_genuine=X_train_np[y_train_np==0]
hist_ae=autoencoder.fit(X_train_genuine,X_train_genuine,
    validation_data=(X_val_np[y_val_np==0],X_val_np[y_val_np==0]),
    epochs=20,batch_size=BATCH_SIZE,callbacks=get_callbacks(),verbose=1)

def reconstruction_errors(model,data):
    rec=model.predict(data,batch_size=BATCH_SIZE,verbose=0)
    return np.mean(np.square(data-rec),axis=1)
ae_val_scores=reconstruction_errors(autoencoder,X_val_np)
ae_test_scores=reconstruction_errors(autoencoder,X_test_np)

## 16. Optimize decision thresholds using validation predictions only

In [ ]:
def choose_threshold(y_true,probs,objective="f1",min_precision=None):
    p,r,t=precision_recall_curve(y_true,probs)
    if len(t)==0: return 0.5
    p,r=p[:-1],r[:-1]
    f1=2*p*r/(p+r+1e-12)
    if min_precision is not None:
        eligible=np.where(p>=min_precision)[0]
        if len(eligible): return float(t[eligible[np.argmax(r[eligible])]])
    idx=int(np.argmax(r if objective=="recall" else f1))
    return float(t[idx])

models={
 "MLP - BCE":(mlp_bce,X_val_np,X_test_np),
 "MLP - Weighted BCE":(mlp_weighted,X_val_np,X_test_np),
 "MLP - Focal Loss":(mlp_focal,X_val_np,X_test_np),
 "CNN - BCE":(cnn_bce,X_val_cnn,X_test_cnn),
 "CNN - Weighted BCE":(cnn_weighted,X_val_cnn,X_test_cnn),
 "CNN - Focal Loss":(cnn_focal,X_val_cnn,X_test_cnn)
}
thresholds={}
for name,(model,xv,xt) in models.items():
    vp=model.predict(xv,batch_size=BATCH_SIZE,verbose=0).ravel()
    thresholds[name]=choose_threshold(y_val_np,vp)
thresholds["Autoencoder"]=choose_threshold(y_val_np,ae_val_scores)
print(thresholds)

## 17. Evaluate every model on the untouched test set

In [ ]:
def evaluate(name,y_true,probs,threshold):
    pred=(probs>=threshold).astype(int)
    print("\\n",name,"threshold:",round(threshold,6))
    print(classification_report(y_true,pred,digits=4,zero_division=0))
    print("Confusion matrix:\\n",confusion_matrix(y_true,pred))
    return {"Model":name,"Threshold":threshold,
        "Precision":precision_score(y_true,pred,zero_division=0),
        "Recall":recall_score(y_true,pred,zero_division=0),
        "F1":f1_score(y_true,pred,zero_division=0),
        "PR-AUC":average_precision_score(y_true,probs),
        "ROC-AUC":roc_auc_score(y_true,probs)},pred

results=[]; test_probs={}; test_preds={}
for name,(model,xv,xt) in models.items():
    prob=model.predict(xt,batch_size=BATCH_SIZE,verbose=0).ravel()
    test_probs[name]=prob
    row,pred=evaluate(name,y_test_np,prob,thresholds[name])
    results.append(row); test_preds[name]=pred

row,pred=evaluate("Autoencoder",y_test_np,ae_test_scores,thresholds["Autoencoder"])
results.append(row); test_preds["Autoencoder"]=pred
results_df=pd.DataFrame(results).sort_values("PR-AUC",ascending=False)
display(results_df)

## 18. Plot confusion matrices and compare metrics

In [ ]:
for name,pred in test_preds.items():
    plt.figure(figsize=(4,3))
    sns.heatmap(confusion_matrix(y_test_np,pred),annot=True,fmt="d",cmap="Blues")
    plt.title("Confusion Matrix: "+name); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.show()

results_df.set_index("Model")[["Precision","Recall","F1","PR-AUC"]].plot(
    kind="bar",figsize=(12,5),ylim=(0,1),rot=35)
plt.title("Test-set comparison"); plt.ylabel("Score"); plt.tight_layout(); plt.show()

## 19. Plot training and validation loss

In [ ]:
histories={"MLP BCE":hist_mlp_bce,"MLP Weighted BCE":hist_mlp_weighted,
 "MLP Focal":hist_mlp_focal,"CNN BCE":hist_cnn_bce,
 "CNN Weighted BCE":hist_cnn_weighted,"CNN Focal":hist_cnn_focal,"Autoencoder":hist_ae}
for name,h in histories.items():
    plt.figure(figsize=(6,3))
    plt.plot(h.history["loss"],label="Training")
    if "val_loss" in h.history: plt.plot(h.history["val_loss"],label="Validation")
    plt.title(name+" loss"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

## 20. Optional: SHAP explanation for best MLP (may take time)

In [ ]:
import shap
mlp_variants={"MLP - BCE":mlp_bce,"MLP - Weighted BCE":mlp_weighted,"MLP - Focal Loss":mlp_focal}
val_ap={n:average_precision_score(y_val_np,m.predict(X_val_np,verbose=0).ravel())
        for n,m in mlp_variants.items()}
best_mlp_name=max(val_ap,key=val_ap.get); best_mlp=mlp_variants[best_mlp_name]
print("Best MLP on validation PR-AUC:",best_mlp_name)
background=X_train_np[np.random.choice(len(X_train_np),min(40,len(X_train_np)),replace=False)]
explain_data=X_test_np[:10]
explainer=shap.KernelExplainer(lambda a:best_mlp.predict(np.asarray(a,dtype=np.float32),verbose=0).ravel(),background)
shap_values=explainer.shap_values(explain_data,nsamples=100)
sv=shap_values[0] if isinstance(shap_values,list) else shap_values
shap.summary_plot(sv,explain_data,feature_names=feature_names)

## 21. Optional: LIME explanation for one transaction

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
lime_explainer=LimeTabularExplainer(X_train_np,feature_names=feature_names,
    class_names=["Genuine","Fraud"],mode="classification")
def predict_classes(a):
    p=best_mlp.predict(np.asarray(a,dtype=np.float32),verbose=0).ravel()
    return np.column_stack([1-p,p])
i=0
exp=lime_explainer.explain_instance(X_test_np[i],predict_classes,num_features=10,num_samples=1000)
print("Actual class:",int(y_test_np[i]))
print("Predicted fraud probability:",float(predict_classes(X_test_np[i:i+1])[0,1]))
exp.show_in_notebook(show_table=True)

## 22. Save trained models and result files

In [ ]:
mlp_bce.save("mlp_bce.keras")
mlp_weighted.save("mlp_weighted_bce.keras")
mlp_focal.save("mlp_focal.keras")
cnn_bce.save("cnn_bce.keras")
cnn_weighted.save("cnn_weighted_bce.keras")
cnn_focal.save("cnn_focal.keras")
autoencoder.save("autoencoder.keras")
results_df.to_csv("model_comparison_results.csv",index=False)
pd.Series(thresholds,name="validation_threshold").to_csv("selected_thresholds.csv")
print("Models and result files saved.")

## 23. Download the results table

In [ ]:
from google.colab import files
files.download("model_comparison_results.csv")

## Notes before reporting
- Only report metrics produced by your own completed run.
- Thresholds are selected on validation data, never on the test set.
- Autoencoder reconstruction error is an anomaly score, not a calibrated fraud probability; interpret its PR-AUC as ranking performance.
- SHAP and LIME explain predictions; they are not prediction models.
- V1–V28 are anonymized PCA components. Do not assign them real-world meanings.
- CNN treats the ordered feature vector as a 1D representation; this is an experiment, not evidence that adjacent PCA components have natural spatial relationships.
- This core notebook includes MLP, CNN and Autoencoder. VAE can be added later if approved and implemented as a separate experiment.